# Notebook pour calculer les hauteurs par rapport au voisinnage
on prend des kernels circulaires de 100m, 500m et 1000m de diamètre

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import xarray
import rioxarray
from dask.distributed import Client
from functools import partial
import time
import shreklib
from scipy.signal import fftconvolve
from os.path import join

In [7]:
def circular_kernel(r):
    xx, yy=np.meshgrid(range(-r,r+1), range(-r, r+1))
    circle=(np.floor(xx**2+yy**2)<=r**2).astype(float)
    circle/=circle.sum()
    return circle

In [8]:
rge_fp="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/rge_snapped_to_grid.tif"

In [ ]:
with rasterio.open(rge_fp, "r") as src:
    mnt=src.read(1)

In [ ]:
mnt_nanned=np.where((np.isnan(mnt)) |( mnt==-99999), 0, mnt)

In [ ]:
mnt_mean_100m=fftconvolve(mnt_nanned, circular_kernel(5), mode="same")

In [ ]:
mnt_mean_500m=fftconvolve(mnt_nanned, circular_kernel(25), mode="same")

In [ ]:
mnt_mean_1000m=fftconvolve(mnt_nanned, circular_kernel(50), mode="same")

In [ ]:
cuvette100m=mnt-mnt_mean_100m
cuvette500m=mnt-mnt_mean_500m
cuvette1000m=mnt-mnt_mean_1000m

In [ ]:
out_path="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/cuvettes"

In [ ]:
for cuvette, size in zip([cuvette100m, cuvette500m, cuvette1000m], [100,500,1000]):
    shreklib.tif_export(cuvette, shreklib.ref_grid["transform"], shreklib.ref_grid["crs"].item(), join(out_path, f"cuvette{size}m.tif"))